In [ ]:
!pip install -q transformers datasets peft trl bitsandbytes accelerate \
             rouge-score evaluate sentencepiece protobuf huggingface_hub

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# SECTION 1: IMPORTS
# ─────────────────────────────────────────────────────────────────────────────
import gc, inspect, json, os, random, textwrap, time, warnings
from typing import Dict, List

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

import torch
from datasets import Dataset, DatasetDict
from rouge_score import rouge_scorer
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    TrainingArguments,
    __version__ as _tv,
)
from peft import (
    LoraConfig,
    PeftModel,
    TaskType,
    get_peft_model,
    prepare_model_for_kbit_training,
)
from trl import SFTTrainer, __version__ as _trlv

warnings.filterwarnings("ignore")

# ─────────────────────────────────────────────────────────────────────────────
# SECTION 2: VERSION HELPERS
# ─────────────────────────────────────────────────────────────────────────────

def _ver(v: str):
    return tuple(int(x) for x in v.split(".")[:3] if x.isdigit())

_EVAL_KEY  = "eval_strategy" if _ver(_tv) >= (4, 41, 0) else "evaluation_strategy"
_SFT_PARAMS = set(inspect.signature(SFTTrainer.__init__).parameters)

print(f"transformers {_tv}  |  trl {_trlv}")
print(f"  TrainingArguments eval key   : {_EVAL_KEY}")
print(f"  SFTTrainer 'tokenizer'       : {'tokenizer' in _SFT_PARAMS}")
print(f"  SFTTrainer 'processing_class': {'processing_class' in _SFT_PARAMS}")

# ─────────────────────────────────────────────────────────────────────────────
# SECTION 3: CONFIG
# ─────────────────────────────────────────────────────────────────────────────

MODEL_NAME   = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
DATASET_PATH = "genai_course_dataset.csv"   # ← upload this CSV to Colab

# ── LoRA config: larger rank + more target modules for stronger adaptation ──
LORA_CFG = dict(
    r=32,                      # was 16  → richer adapter
    lora_alpha=64,             # was 32  → alpha = 2*r (standard)
    lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],   # all linear layers
    bias="none",
    task_type=TaskType.CAUSAL_LM,
)

TRAIN_CFG = dict(
    output_dir="./finetune_output",
    num_train_epochs=3,                  # was 2 → better convergence
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,       # effective batch = 8
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.06,                   # ~6% warm-up
    max_seq_length=512,
    logging_steps=10,
    save_strategy="epoch",
    fp16=False,
    bf16=False,                          # T4 does not support bf16
    optim="paged_adamw_32bit",
    report_to="none",
)

USE_4BIT = True   # QLoRA — required for Colab free T4

# ─────────────────────────────────────────────────────────────────────────────
# SECTION 4: DATASET  — load from CSV
# ─────────────────────────────────────────────────────────────────────────────

SYSTEM_MSG = ("You are a helpful university course assistant specializing in "
              "Generative AI and Large Language Models. "
              "Answer questions accurately and concisely.")


def load_csv_dataset(path: str) -> pd.DataFrame:
    """Load the provided CSV. Expects columns: instruction, output."""
    df = pd.read_csv(path)
    assert "instruction" in df.columns and "output" in df.columns, \
        "CSV must have columns 'instruction' and 'output'"
    df = df.dropna(subset=["instruction", "output"])
    df = df[df["instruction"].str.strip().ne("") & df["output"].str.strip().ne("")]
    print(f"  Loaded {len(df)} rows from {path}")
    return df.reset_index(drop=True)


def format_row(row) -> Dict:
    """Convert a CSV row to TinyLlama chat format."""
    return {"text": (f"<|system|>\n{SYSTEM_MSG}</s>\n"
                     f"<|user|>\n{row['instruction']}</s>\n"
                     f"<|assistant|>\n{row['output']}</s>\n")}


def make_hf_dataset(df: pd.DataFrame, val_split: float = 0.1) -> DatasetDict:
    rows = [format_row(r) for _, r in df.iterrows()]
    random.shuffle(rows)
    cut  = int(len(rows) * (1 - val_split))
    return DatasetDict({
        "train":      Dataset.from_list(rows[:cut]),
        "validation": Dataset.from_list(rows[cut:]),
    })

# ─────────────────────────────────────────────────────────────────────────────
# SECTION 5: MODEL LOADING
# ─────────────────────────────────────────────────────────────────────────────

def load_model_and_tokenizer(model_name: str, use_4bit: bool = True):
    print(f"\nLoading tokenizer: {model_name}")
    tok = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
    tok.pad_token    = tok.eos_token
    tok.padding_side = "right"

    print("Loading model…")
    if use_4bit:
        bnb = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_use_double_quant=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch.bfloat16,
        )
        model = AutoModelForCausalLM.from_pretrained(
            model_name, quantization_config=bnb,
            device_map="auto", trust_remote_code=True)
        model = prepare_model_for_kbit_training(model)
        print("  QLoRA 4-bit NF4 enabled")
    else:
        model = AutoModelForCausalLM.from_pretrained(
            model_name, torch_dtype=torch.float16, device_map="auto")

    total  = sum(p.numel() for p in model.parameters())
    print(f"  Total params: {total:,}")
    return model, tok

# ─────────────────────────────────────────────────────────────────────────────
# SECTION 6: TRAINING  — version-safe SFTTrainer
# ─────────────────────────────────────────────────────────────────────────────

def train_model(model, tokenizer, dataset: DatasetDict, cfg: Dict):
    total_train  = len(dataset["train"])
    eff_batch    = cfg["per_device_train_batch_size"] * cfg["gradient_accumulation_steps"]
    total_steps  = int(total_train / eff_batch) * cfg["num_train_epochs"]
    warmup_steps = max(1, int(total_steps * cfg["warmup_ratio"]))
    print(f"  Total steps: {total_steps}  |  Warmup: {warmup_steps}")

    training_args = TrainingArguments(
        output_dir=cfg["output_dir"],
        num_train_epochs=cfg["num_train_epochs"],
        per_device_train_batch_size=cfg["per_device_train_batch_size"],
        gradient_accumulation_steps=cfg["gradient_accumulation_steps"],
        learning_rate=cfg["learning_rate"],
        lr_scheduler_type=cfg["lr_scheduler_type"],
        warmup_steps=warmup_steps,
        logging_steps=cfg["logging_steps"],
        save_strategy=cfg["save_strategy"],
        fp16=cfg["fp16"],
        bf16=cfg["bf16"],
        optim=cfg["optim"],
        report_to=cfg["report_to"],
        **{_EVAL_KEY: "epoch"},
    )

    sft_kwargs = dict(
        model=model,
        train_dataset=dataset["train"],
        eval_dataset=dataset["validation"],
        args=training_args,
        # # max_seq_length=cfg["max_seq_length"],
        # dataset_text_field="text",
        # packing=False,                   # more stable on small GPU
    )

    if "tokenizer" in _SFT_PARAMS:
        sft_kwargs["tokenizer"] = tokenizer
    elif "processing_class" in _SFT_PARAMS:
        sft_kwargs["processing_class"] = tokenizer

    trainer = SFTTrainer(**sft_kwargs)

    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total     = sum(p.numel() for p in model.parameters())
    print(f"  Trainable: {trainable:,} / {total:,} ({100*trainable/total:.2f}%)")

    print("\n" + "="*60)
    print("STARTING FINE-TUNING")
    print("="*60)
    t0     = time.time()
    result = trainer.train()
    elapsed = time.time() - t0
    print(f"\nDone in {elapsed/60:.1f} min | Final loss: {result.training_loss:.4f}")

    trainer.model.save_pretrained(cfg["output_dir"])
    tokenizer.save_pretrained(cfg["output_dir"])

    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    gc.collect()
    return trainer, result, elapsed

# ─────────────────────────────────────────────────────────────────────────────
# SECTION 7: EVALUATION
# ─────────────────────────────────────────────────────────────────────────────

def generate_response(model, tokenizer, question: str,
                       max_new_tokens: int = 300) -> str:
    prompt = (f"<|system|>\n{SYSTEM_MSG}</s>\n"
              f"<|user|>\n{question}</s>\n"
              f"<|assistant|>\n")
    inp = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(
            **inp,
            max_new_tokens=max_new_tokens,
            temperature=0.6,        # slightly lower → more focused answers
            do_sample=True,
            repetition_penalty=1.15,  # prevents looping
            pad_token_id=tokenizer.eos_token_id,
        )
    new_tokens = out[0][inp["input_ids"].shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()


def compute_rouge(preds: List[str], refs: List[str]) -> Dict:
    sc  = rouge_scorer.RougeScorer(["rouge1", "rouge2", "rougeL"], use_stemmer=True)
    agg = {"rouge1": [], "rouge2": [], "rougeL": []}
    for p, r in zip(preds, refs):
        s = sc.score(r, p)
        for k in agg:
            agg[k].append(s[k].fmeasure)
    return {k: float(np.mean(v)) for k, v in agg.items()}


# ── Evaluation questions (held-out — not in training set) ────────────────────
EVAL_QA = [
    ("What is a VAE and how is it trained?",
     "A VAE encodes inputs into a probabilistic latent space using mean and variance vectors "
     "and decodes samples to reconstruct them. Training minimizes the ELBO — reconstruction "
     "loss plus KL divergence — using the reparameterization trick to allow backpropagation "
     "through the sampling step."),
    ("What are the main LLM evaluation metrics?",
     "Perplexity (intrinsic), BLEU (translation n-gram precision), ROUGE (summarization n-gram "
     "recall and F1), BERTScore (semantic similarity via contextual embeddings), and human "
     "evaluation for open-ended generation tasks."),
    ("Why is positional encoding needed in Transformers?",
     "Self-attention is permutation-invariant so it cannot distinguish token order. Positional "
     "encoding adds sequence order information via sinusoidal functions or learned embeddings "
     "summed with the token embeddings before the first attention layer."),
    ("What is the reparameterization trick?",
     "Instead of sampling z directly from N(mu, sigma^2), write z = mu + sigma * epsilon where "
     "epsilon ~ N(0,1). Gradients flow through the deterministic mu and sigma; epsilon carries "
     "the stochasticity, enabling backpropagation through the sampling step."),
    ("What is mode collapse in GANs?",
     "Mode collapse occurs when the generator produces only a small subset of data modes, "
     "ignoring diversity. Solutions include Wasserstein GAN with gradient penalty, mini-batch "
     "discrimination, progressive growing, and spectral normalization."),
    ("What is the difference between RAG and fine-tuning?",
     "RAG retrieves external knowledge at inference without changing model weights, ideal for "
     "frequently updated facts. Fine-tuning updates weights on domain data, ideal for style "
     "and behavioral changes. They are complementary and can be combined."),
    ("Explain LoRA in detail.",
     "LoRA injects trainable low-rank matrices alongside frozen pretrained weights. For weight "
     "W in R^(d x k), it adds delta_W = BA where B is R^(d x r) and A is R^(r x k). Only A "
     "and B are trained, reducing trainable parameters by orders of magnitude while achieving "
     "near full fine-tuning quality."),
    ("What is RLHF?",
     "RLHF aligns LLMs with human preferences in three stages: supervised fine-tuning on "
     "human demonstrations, reward model training on preference pairs, and PPO reinforcement "
     "learning guided by the reward model with a KL penalty to prevent reward hacking."),
]


def run_evaluation(base_model, tokenizer, output_dir: str):
    print("\n" + "="*60)
    print("EVALUATION: BASE vs FINE-TUNED")
    print("="*60)

    ft_model = PeftModel.from_pretrained(base_model, output_dir)
    ft_model.eval()

    sc_l = rouge_scorer.RougeScorer(["rougeL"], use_stemmer=True)
    rows, b_preds, f_preds, refs = [], [], [], []

    for q, ref in EVAL_QA:
        print(f"\nQ: {q[:65]}…")

        ft_model.disable_adapter_layers()
        b_ans = generate_response(ft_model, tokenizer, q)
        b_preds.append(b_ans)
        refs.append(ref)
        b_s = sc_l.score(ref, b_ans)["rougeL"].fmeasure
        print(f"  Base [{b_s:.3f}]: {b_ans[:90]}…")

        ft_model.enable_adapter_layers()
        f_ans = generate_response(ft_model, tokenizer, q)
        f_preds.append(f_ans)
        f_s = sc_l.score(ref, f_ans)["rougeL"].fmeasure
        print(f"  FT   [{f_s:.3f}]: {f_ans[:90]}…")

        rows.append({
            "question":        q,
            "reference":       ref,
            "base_answer":     b_ans,
            "finetuned_answer": f_ans,
            "base_rougeL":     round(b_s, 4),
            "ft_rougeL":       round(f_s, 4),
            "improvement":     round(f_s - b_s, 4),
        })

    base_r = compute_rouge(b_preds, refs)
    ft_r   = compute_rouge(f_preds, refs)

    print("\n" + "="*60)
    print(f"{'Metric':<10} {'Base':>8} {'FT':>8} {'Delta':>8}")
    print("-"*40)
    for m in ["rouge1", "rouge2", "rougeL"]:
        delta = ft_r[m] - base_r[m]
        print(f"{m:<10} {base_r[m]:>8.4f} {ft_r[m]:>8.4f} {delta:>+8.4f}")

    del ft_model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    gc.collect()
    return pd.DataFrame(rows), base_r, ft_r

# ─────────────────────────────────────────────────────────────────────────────
# SECTION 8: VISUALIZATION
# ─────────────────────────────────────────────────────────────────────────────

def create_visualizations(trainer, df_eval: pd.DataFrame,
                           base_r: Dict, ft_r: Dict,
                           elapsed: float, cfg: Dict):
    fig = plt.figure(figsize=(18, 11))
    fig.suptitle("Fine-Tuning Analysis — QLoRA TinyLlama-1.1B | CS601",
                 fontsize=13, fontweight="bold")
    gs = gridspec.GridSpec(2, 3, figure=fig, hspace=0.48, wspace=0.40)

    # ── 1. Training loss from trainer log ────────────────────────────────────
    ax1 = fig.add_subplot(gs[0, 0])
    log_history = trainer.state.log_history
    steps_loss  = [(e["step"], e["loss"]) for e in log_history if "loss" in e]
    if steps_loss:
        xs, ys = zip(*steps_loss)
        ax1.plot(xs, ys, "#e74c3c", lw=2, label="Train loss")
    eval_loss = [(e["step"], e["eval_loss"]) for e in log_history if "eval_loss" in e]
    if eval_loss:
        xe, ye = zip(*eval_loss)
        ax1.plot(xe, ye, "#3498db", lw=2, ls="--", label="Val loss")
        ax1.legend(fontsize=8)
    ax1.set(xlabel="Step", ylabel="Loss", title="Training & Validation Loss")
    ax1.grid(alpha=0.3)

    # ── 2. ROUGE comparison ──────────────────────────────────────────────────
    ax2 = fig.add_subplot(gs[0, 1])
    ms  = ["rouge1", "rouge2", "rougeL"]
    x, w = np.arange(3), 0.35
    ax2.bar(x - w/2, [base_r[m] for m in ms], w, label="Base",       color="#95a5a6")
    ax2.bar(x + w/2, [ft_r[m]   for m in ms], w, label="Fine-Tuned", color="#2ecc71")
    ax2.set_xticks(x); ax2.set_xticklabels(["ROUGE-1", "ROUGE-2", "ROUGE-L"])
    ax2.set(ylabel="F1 Score", title="ROUGE: Base vs Fine-Tuned")
    ax2.legend(fontsize=8)
    for i, m in enumerate(ms):
        delta = ft_r[m] - base_r[m]
        ax2.annotate(f"{delta:+.3f}", (i + w/2, ft_r[m] + 0.002),
                     ha="center", fontsize=7, color="#27ae60")

    # ── 3. Per-question improvement ──────────────────────────────────────────
    ax3 = fig.add_subplot(gs[0, 2])
    colors3 = ["#2ecc71" if v >= 0 else "#e74c3c" for v in df_eval["improvement"]]
    bars = ax3.bar([f"Q{i+1}" for i in range(len(df_eval))],
                   df_eval["improvement"], color=colors3)
    ax3.axhline(0, color="k", lw=0.8, ls="--")
    ax3.set(ylabel="ROUGE-L Improvement", title="Per-Question ROUGE-L Improvement")
    ax3.grid(alpha=0.3, axis="y")
    mean_imp = df_eval["improvement"].mean()
    ax3.axhline(mean_imp, color="#f39c12", lw=1.5, ls="-",
                label=f"Mean Δ = {mean_imp:+.3f}")
    ax3.legend(fontsize=8)

    # ── 4. Parameter efficiency ──────────────────────────────────────────────
    ax4 = fig.add_subplot(gs[1, 0])
    for label, tp, gb, col in [
        ("Full FT (7B)",  100.0,  28, "#e74c3c"),
        ("LoRA r=16",       0.65, 14, "#f39c12"),
        ("Our QLoRA r=32",  0.85,  6, "#2ecc71"),
    ]:
        ax4.scatter(tp, gb, s=260, c=col, zorder=5, edgecolors="white", lw=0.8)
        ax4.annotate(label, (tp, gb), xytext=(5, 4),
                     textcoords="offset points", fontsize=8)
    ax4.set(xlabel="Trainable Params (%)", ylabel="GPU Memory (GB)",
            title="Parameter Efficiency")
    ax4.grid(alpha=0.3)

    # ── 5. Answer length comparison ──────────────────────────────────────────
    ax5 = fig.add_subplot(gs[1, 1])
    bl  = [len(a.split()) for a in df_eval["base_answer"]]
    fl  = [len(a.split()) for a in df_eval["finetuned_answer"]]
    idx = list(range(len(df_eval)))
    ax5.bar([i - 0.2 for i in idx], bl, 0.4, label="Base",       color="#95a5a6")
    ax5.bar([i + 0.2 for i in idx], fl, 0.4, label="Fine-Tuned", color="#3498db")
    ax5.set_xticks(idx); ax5.set_xticklabels([f"Q{i+1}" for i in idx])
    ax5.set(ylabel="Words", title="Answer Length Comparison")
    ax5.legend(fontsize=8)

    # ── 6. Config summary ────────────────────────────────────────────────────
    ax6 = fig.add_subplot(gs[1, 2]); ax6.axis("off")
    eff_batch = cfg["per_device_train_batch_size"] * cfg["gradient_accumulation_steps"]
    txt = (
        f"  Config Summary\n"
        f"  {'─'*30}\n"
        f"  Model        : TinyLlama-1.1B-Chat\n"
        f"  Quantization : 4-bit NF4 (QLoRA)\n"
        f"  GPU RAM      : ~6 GB\n\n"
        f"  LoRA rank    : {cfg.get('r', LORA_CFG['r'])}\n"
        f"  LoRA alpha   : {cfg.get('lora_alpha', LORA_CFG['lora_alpha'])}\n"
        f"  Targets      : q/k/v/o + gate/up/down\n"
        f"  Trainable    : ~0.85%\n\n"
        f"  Dataset      : 1500 examples (CSV)\n"
        f"  Epochs       : {cfg['num_train_epochs']}\n"
        f"  Eff. batch   : {eff_batch}\n"
        f"  LR           : {cfg['learning_rate']}\n"
        f"  Optimizer    : paged_adamw_32bit\n"
        f"  Train time   : {elapsed/60:.1f} min\n\n"
        f"  trl          : {_trlv}\n"
        f"  transformers : {_tv}"
    )
    ax6.text(0.05, 0.97, txt, transform=ax6.transAxes, fontsize=8.5, va="top",
             fontfamily="monospace",
             bbox=dict(boxstyle="round", facecolor="#f0f0f0", alpha=0.75))
    ax6.set_title("Configuration", fontweight="bold")

    plt.savefig("finetuning_analysis.png", dpi=130, bbox_inches="tight")
    plt.show()
    plt.close(fig)
    print("Saved: finetuning_analysis.png")
    gc.collect()

# ─────────────────────────────────────────────────────────────────────────────
# SECTION 9: MAIN
# ─────────────────────────────────────────────────────────────────────────────

def main():
    print("\n" + "="*70)
    print("ASSIGNMENT 2: FINE-TUNING LLM — CS601  (Improved)")
    print("="*70)

    # 1. Load dataset from CSV
    print(f"\n[1] Loading dataset from {DATASET_PATH}…")
    df = load_csv_dataset(DATASET_PATH)
    print(f"  {len(df)} examples loaded")
    print(f"  Avg instruction length : {df['instruction'].str.split().str.len().mean():.1f} words")
    print(f"  Avg output length      : {df['output'].str.split().str.len().mean():.1f} words")

    # 2. Load model
    print("\n[2] Loading base model…")
    model, tokenizer = load_model_and_tokenizer(MODEL_NAME, USE_4BIT)

    # 3. Prepare HF dataset
    print("\n[3] Preparing HuggingFace dataset…")
    hf_ds = make_hf_dataset(df, val_split=0.1)
    print(f"  Train: {len(hf_ds['train'])}  |  Val: {len(hf_ds['validation'])}")

    # 4. Apply LoRA
    print("\n[4] Applying LoRA…")
    lora_config = LoraConfig(**LORA_CFG)
    model = get_peft_model(model, lora_config)
    model.print_trainable_parameters()

    # 5. Train
    print("\n[5] Training…")
    trainer, result, elapsed = train_model(model, tokenizer, hf_ds,
                                            {**TRAIN_CFG, **LORA_CFG})

    # 6. Evaluate
    print("\n[6] Evaluating…")
    df_eval, base_r, ft_r = run_evaluation(model, tokenizer, TRAIN_CFG["output_dir"])
    df_eval.to_csv("finetuning_results.csv", index=False)
    print("  Saved: finetuning_results.csv")

    # 7. Visualize
    print("\n[7] Creating visualizations…")
    create_visualizations(trainer, df_eval, base_r, ft_r, elapsed,
                           {**TRAIN_CFG, **LORA_CFG})

    # 8. Demo
    print("\n" + "="*70)
    print("DEMO: BASE vs FINE-TUNED")
    print("="*70)
    demo_questions = [
        "What is Chain-of-Thought prompting and when should I use it?",
        "Explain QLoRA and why it is useful for fine-tuning on limited GPU memory.",
    ]

    ft_m = PeftModel.from_pretrained(model, TRAIN_CFG["output_dir"])
    ft_m.eval()

    for demo_q in demo_questions:
        ft_m.disable_adapter_layers()
        b_ans = generate_response(ft_m, tokenizer, demo_q)

        ft_m.enable_adapter_layers()
        f_ans = generate_response(ft_m, tokenizer, demo_q)

        print(f"\nQ: {demo_q}")
        print(f"\nBase Model:\n{textwrap.fill(b_ans, 72)}")
        print(f"\nFine-Tuned:\n{textwrap.fill(f_ans, 72)}")
        print("-" * 72)

    del ft_m, model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    gc.collect()

    print("\n" + "="*70)
    print(f"COMPLETE — training time: {elapsed/60:.1f} min")
    print("Outputs: finetuning_results.csv  finetuning_analysis.png")
    return df_eval, base_r, ft_r


if __name__ == "__main__":
    df_eval, base_r, ft_r = main()